# Đánh giá các bộ siêu tham số đã chọn — CluStream & DenStream

Notebook này lấy **một bộ siêu tham số cho mỗi hướng** mà pilot `hpo_pipeline.hpo_compare` đã chọn, rồi chạy các bộ đó trên **KDDCup99 (10%)**, **Covertype** và **INSECTS** để xem việc tune có làm chất lượng tốt hơn không, và phải trả giá bao nhiêu tài nguyên.

| Hướng | Bộ được chọn thế nào |
|---|---|
| **default** | default của constructor River |
| **arm 1** | grid thô, bộ có điểm gộp tốt nhất trên cả 5 dataset tune |
| **arm 2** | grid thô, LODO; bộ được nhiều vòng chọn nhất |
| **arm 3** | grid thay thế (`epsilon = c·s(D)`, `max_micro_clusters = micro_ratio·k`), LODO |
| **P2R R_knee** | hình dạng cố định; mức độ mịn đọc trên đường cong `matr_trace_win` (không nhãn) của từng dataset |

**Đường dữ liệu** giống pilot: dữ liệu thô → xáo trộn → luồng, **không scaler**. Hoán vị gắn với seed (`shuffle_seed = 1000 + seed`). Seed 1 đã dùng để tune, nên mặc định chạy seed 2 và 3.

**Mỗi run là một tiến trình con** (`hpo_pipeline.final_eval`), vì CluStream (KMeans bên trong) chỉ tái lập được khi `PYTHONHASHSEED` cố định — kernel đang chạy không đổi được giá trị này. Kết quả ghi dần ra JSONL; chạy lại cell thì run đã xong được bỏ qua.

> ⚠️ **Cách đọc đúng.** Ba dataset này nằm trong 5 dataset đã dùng để chọn bộ, nên điểm ở đây là **trong mẫu** cho mọi hướng. Con số trung thực cho dataset chưa thấy là ước lượng LODO trong `hpo_pipeline/out/compare/report_50000.md`. Notebook này trả lời câu khác: *chạy nguyên độ dài dataset thì bộ nào tốt hơn default, và tốn bao nhiêu.*

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Notebook nằm ở gốc repo framework; hpo_pipeline cần import được từ đây.
REPO = Path.cwd()
if not (REPO / "hpo_pipeline").is_dir():
    raise SystemExit(f"Hãy mở notebook từ gốc river-customized-benchmark-framework (cwd={REPO})")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from hpo_pipeline import final_eval as fe  # noqa: E402

pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1. Bảng điều khiển

Mọi thứ cần chỉnh nằm ở cell dưới.

- `N_SAMPLES = None` chạy **toàn bộ** dataset: KDDCup99 10% có 494.021 mẫu, Covertype 581.012, INSECTS 79.986. CluStream trên hai dataset lớn mất cỡ 30–70 phút mỗi run; tổng cộng vài giờ với 8–12 worker.
- Muốn xem nhanh, đặt `N_SAMPLES = 50_000` (đúng độ dài lúc tune) hoặc `2_000` (chạy thử).
- `FINAL_SETS_FILE`: `final_sets_50000_headroom.json` (gộp theo headroom — mặc định của `hpo_compare`), `final_sets_50000.json` (trung bình ARI) hoặc `final_sets_50000_rank.json` (thứ hạng). Các arm irace (`irace_raw`, `irace_ladder`, `p2r_irace_*`) nằm trong cùng file khi chạy `hpo_compare --irace ...`.

In [ ]:
FINAL_SETS_FILE = REPO / "hpo_pipeline/out/compare/final_sets_50000_headroom.json"
ALGOS = ["CluStream", "DenStream"]
DATASETS = ["kdd99", "covtype", "insects_ir_bal"]   # kdd99 = KDDCup99 10%, 34 thuộc tính liên tục
ARMS = ["default", "arm1_raw_all", "arm2_raw_lodo", "arm3_sf_lodo", "p2r_R_knee_matr_trace_win"]
SEEDS = [2, 3, 4, 5, 6]          # = hpo_compare.FINAL_SEEDS; disjoint from TUNE_SEEDS
N_SAMPLES = None          # None = toàn bộ dataset
WORKERS = 8
TIMEOUT_S = 4 * 3600      # giới hạn mỗi run
RESULTS = REPO / "hpo_pipeline/out/final_eval" / f"runs_{Path(FINAL_SETS_FILE).stem}.jsonl"

ARM_LABEL = {a: fe.ARM_NAMES.get(a, a) for a in ARMS}
ARM_COLOR = dict(zip(ARMS, ["#8c8c8c", "#1f77b4", "#6baed6", "#d62728", "#2ca02c"]))
# arm 1 và arm 2 thường ra cùng một bộ; nét đứt giúp thấy cả hai khi đường trùng nhau.
ARM_STYLE = {a: ("--" if a == "arm2_raw_lodo" else "-") for a in ARMS}
DATASET_LABEL = {"kdd99": "KDDCup99 10%", "covtype": "Covertype", "insects_ir_bal": "INSECTS"}

## 2. Các bộ sẽ chạy

Mỗi dòng là một bộ. Nếu arm 1 và arm 2 trùng bộ (các vòng LODO đồng thuận với lựa chọn trên mọi dataset), hai hướng dùng chung run và các đường của chúng chồng lên nhau — arm 2 vẽ nét đứt.

Mỗi bộ viết theo đơn vị của grid (`epsilon = c · s(D)` là **một** bộ dù giá trị tuyệt đối khác nhau giữa dataset). Bảng thứ hai là giá trị tuyệt đối thực sự đưa vào River trên từng dataset.

In [ ]:
final_sets = fe.load_final_sets(FINAL_SETS_FILE)

rows = []
for algo in ALGOS:
    for arm in ARMS:
        e = final_sets.get(algo, {}).get(arm)
        if e is None:
            rows.append({"algo": algo, "arm": ARM_LABEL[arm], "set": "— không có —"})
            continue
        how = f"{e['votes']}/{e['rounds']} phiếu" if e.get("votes") is not None else ""
        rows.append({"algo": algo, "arm": ARM_LABEL[arm], "set": e["set"], "chọn bởi": how})
display(pd.DataFrame(rows).set_index(["algo", "arm"]))

TUNED = {"CluStream": ["n_macro_clusters", "max_micro_clusters", "micro_cluster_r_factor"],
         "DenStream": ["epsilon", "mu", "beta"]}
rows = []
for algo in ALGOS:
    for arm in ARMS:
        e = final_sets.get(algo, {}).get(arm) or {}
        for ds in DATASETS:
            p = e.get("resolved", {}).get(ds)
            rows.append({"algo": algo, "arm": ARM_LABEL[arm], "dataset": DATASET_LABEL[ds],
                         **({k: p.get(k) for k in TUNED[algo]} if p else {})})
resolved = pd.DataFrame(rows)
for algo in ALGOS:
    display(Markdown(f"**{algo}** — giá trị tuyệt đối trên từng dataset"))
    display(resolved[resolved.algo == algo].drop(columns="algo")
            .dropna(axis=1, how="all").set_index(["arm", "dataset"]))

## 3. Chạy

Các hướng có cùng tham số trên một dataset (ví dụ arm 1 và arm 2) dùng chung một run. Cell này chạy lại được: run đã có trong `RESULTS` được bỏ qua; run lỗi hoặc quá giờ chỉ chạy lại khi đặt `RETRY_FAILED = True`.

In [ ]:
RETRY_FAILED = False

jobs, missing = fe.build_jobs(final_sets, ALGOS, DATASETS, ARMS, SEEDS, N_SAMPLES)
for m in missing:
    print(f"không có bộ: {m['algo']} / {m['arm']} / {m['dataset']} ({m['reason']})")
print(f"{len(jobs)} run cho {len(ALGOS)} thuật toán × {len(ARMS)} hướng × "
      f"{len(DATASETS)} dataset × {len(SEEDS)} seed (đã gộp run trùng tham số)")

fe.run_jobs(jobs, RESULTS, workers=WORKERS, timeout=TIMEOUT_S, retry_failed=RETRY_FAILED)

In [ ]:
records = fe.load_records(jobs, RESULTS)
summary = fe.summary_frame(records)
windows = fe.window_frame(records, "ari")
checkpoints = fe.checkpoint_frame(records)

ARM_ORDER = [ARM_LABEL[a] for a in ARMS]
summary["arm_label"] = pd.Categorical(summary.arm.map(ARM_LABEL), categories=ARM_ORDER, ordered=True)
status = (summary.pivot_table(index=["algo", "arm_label"], columns="dataset", values="status",
                              aggfunc=lambda s: ",".join(sorted(set(s))), observed=True)
          .reindex(columns=DATASETS).rename(columns=DATASET_LABEL))
display(Markdown("**Trạng thái run** (mọi ô phải là `ok` trước khi đọc biểu đồ)"))
display(status)

ok = summary[summary.status == "ok"].copy()

## 4. Chất lượng

ARI là chỉ số chính (như lúc tune); NMI và purity để đối chiếu. Cột là trung bình qua các seed, vạch đen là khoảng min–max giữa các seed — nếu vạch dài hơn khoảng cách giữa hai cột thì chênh lệch đó chưa đọc được.

Có **hai** cột purity và chúng khác nhau: `purity` là bản có trọng số theo mẫu (micro), còn `purity_macro` là trung bình **không trọng số qua các cụm** — đúng công thức mà DenStream (SDM 2006 §5), RepStream (Eq. 18) và SOStream in ra. Dùng `purity_macro` khi muốn đối chiếu với số trong các paper đó; cả hai đều tăng cơ học theo số cụm nên chỉ đọc được khi có `n_predicted_clusters` bên cạnh.


In [ ]:
def bar_panel(frame, metric, title, ylabel=None):
    fig, axes = plt.subplots(1, len(ALGOS), figsize=(6.5 * len(ALGOS), 4.2), squeeze=False)
    width = 0.8 / len(ARMS)
    for ax, algo in zip(axes[0], ALGOS):
        sub = frame[frame.algo == algo]
        x = np.arange(len(DATASETS))
        for i, arm in enumerate(ARMS):
            g = sub[sub.arm == arm].groupby("dataset")[metric]
            mean = g.mean().reindex(DATASETS)
            lo, hi = g.min().reindex(DATASETS), g.max().reindex(DATASETS)
            pos = x - 0.4 + width * (i + 0.5)
            ax.bar(pos, mean, width, color=ARM_COLOR[arm], label=ARM_LABEL[arm])
            # mean - lo va hi - mean khong am theo dinh nghia, nhung trung binh
            # float64 cua nhieu gia tri GIONG HET nhau co the tut xuong duoi chung
            # 1 ULP: 0% so truong hop khi n=2, ~6% khi n=5. Vi the loi nay chi xuat
            # hien tu khi FINAL_SEEDS di tu 2 len 5 seed. matplotlib tu choi moi
            # yerr am ke ca -1e-16, nen cat phan nhieu do di. NaN duoc giu nguyen
            # (np.clip khong dung den no) vi matplotlib cho phep NaN.
            err_lo = np.clip((mean - lo).to_numpy(), 0.0, None)
            err_hi = np.clip((hi - mean).to_numpy(), 0.0, None)
            ax.errorbar(pos, mean, yerr=[err_lo, err_hi], fmt="none", ecolor="k",
                        elinewidth=0.8, capsize=2)
        ax.set_xticks(x, [DATASET_LABEL[d] for d in DATASETS])
        ax.axhline(0, color="k", lw=0.6)
        ax.set_title(algo)
        ax.set_ylabel(ylabel or metric)
    axes[0][0].legend(fontsize=8, loc="best")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


bar_panel(ok, "ari", "ARI toàn luồng — cao hơn là tốt hơn")
bar_panel(ok, "nmi", "NMI toàn luồng")
bar_panel(ok, "purity", "Purity toàn luồng")

### 4.1 Có tốt hơn default không?

Hiệu ARI so với default trên cùng dataset và cùng seed, rồi lấy trung bình qua seed. Dương (xanh) là tốt hơn default.

In [ ]:
base = ok[ok.arm == "default"].set_index(["algo", "dataset", "seed"])["ari"]
delta = ok[ok.arm != "default"].copy()
delta["d_ari"] = delta["ari"].to_numpy() - base.reindex(
    pd.MultiIndex.from_frame(delta[["algo", "dataset", "seed"]])).to_numpy()
table = delta.pivot_table(index=["algo", "arm_label"], columns="dataset", values="d_ari",
                          observed=True)
table = table.reindex(columns=[d for d in DATASETS if d in table.columns])
table["mean"] = table.mean(axis=1)
table = table.rename(columns=DATASET_LABEL)

fig, ax = plt.subplots(figsize=(8, 0.45 * len(table) + 1.2))
lim = np.nanmax(np.abs(table.to_numpy())) or 1.0
im = ax.imshow(table.to_numpy(dtype=float), cmap="RdYlGn", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(table.shape[1]), table.columns)
ax.set_yticks(range(table.shape[0]), [f"{a} · {b}" for a, b in table.index])
for (i, j), v in np.ndenumerate(table.to_numpy(dtype=float)):
    ax.text(j, i, "–" if np.isnan(v) else f"{v:+.3f}", ha="center", va="center", fontsize=8)
ax.grid(False)
fig.colorbar(im, ax=ax, label="ARI − ARI default")
ax.set_title("Hiệu ARI so với default")
fig.tight_layout()
plt.show()
display(table.round(4))

### 4.2 Chất lượng theo thời gian

Trái: ARI **trong từng cửa sổ** 1.000 mẫu (nhìn hành vi dọc luồng). Phải: ARI **cộng dồn** từ đầu luồng (đường hội tụ; điểm cuối chính là ARI toàn luồng). Trung bình qua seed; cửa sổ được làm mượt bằng trung bình trượt để dễ nhìn.

In [ ]:
SMOOTH = 10  # số cửa sổ trong trung bình trượt

for algo in ALGOS:
    fig, axes = plt.subplots(len(DATASETS), 2, figsize=(13, 3.2 * len(DATASETS)), squeeze=False)
    for r, ds in enumerate(DATASETS):
        sub = windows[(windows.algo == algo) & (windows.dataset == ds)]
        for arm in ARMS:
            g = sub[sub.arm == arm].groupby("position")[["window", "cumulative"]].mean()
            if g.empty:
                continue
            kw = dict(color=ARM_COLOR[arm], label=ARM_LABEL[arm], lw=1.2, ls=ARM_STYLE[arm])
            axes[r][0].plot(g.index, g["window"].rolling(SMOOTH, min_periods=1).mean(), **kw)
            axes[r][1].plot(g.index, g["cumulative"], **kw)
        axes[r][0].set_ylabel(f"{DATASET_LABEL[ds]}\nARI")
    axes[0][0].set_title("ARI theo cửa sổ (trượt)")
    axes[0][1].set_title("ARI cộng dồn")
    for ax in axes[-1]:
        ax.set_xlabel("số mẫu đã xử lý")
    axes[0][1].legend(fontsize=8)
    fig.suptitle(algo)
    fig.tight_layout()
    plt.show()

## 5. Tài nguyên (usage)

- **Thông lượng** (mẫu/giây) và **thời gian chạy**: chi phí CPU.
- **Bộ nhớ mô hình**: đỉnh bộ nhớ của riêng mô hình River (không tính cả tiến trình).
- **Số micro-cluster**: nguồn gốc của phần lớn chi phí thời gian và bộ nhớ.

Thời gian đo trong tiến trình con, chỉ gồm `predict_one` + `learn_one`; các run chạy song song nên có sai lệch nhỏ do tranh chấp CPU.

In [ ]:
bar_panel(ok, "throughput_sps", "Thông lượng — cao hơn là tốt hơn", "mẫu / giây")
bar_panel(ok, "runtime_s", "Tổng thời gian chạy — thấp hơn là tốt hơn", "giây")
bar_panel(ok, "peak_memory_mb", "Đỉnh bộ nhớ mô hình — thấp hơn là tốt hơn", "MB")
bar_panel(ok, "max_micro_clusters", "Số micro-cluster lớn nhất", "micro-cluster")

In [ ]:
CURVES = [("runtime_s", "thời gian cộng dồn (s)"),
          ("memory_mb", "bộ nhớ mô hình (MB)"),
          ("n_micro_clusters", "số micro-cluster")]

for algo in ALGOS:
    fig, axes = plt.subplots(len(DATASETS), len(CURVES),
                             figsize=(5.2 * len(CURVES), 3.1 * len(DATASETS)), squeeze=False)
    for r, ds in enumerate(DATASETS):
        sub = checkpoints[(checkpoints.algo == algo) & (checkpoints.dataset == ds)]
        for c, (col, label) in enumerate(CURVES):
            ax = axes[r][c]
            for arm in ARMS:
                g = sub[sub.arm == arm].groupby("n_seen")[col].mean().dropna()
                if not g.empty:
                    ax.plot(g.index, g.to_numpy(), color=ARM_COLOR[arm], label=ARM_LABEL[arm],
                            lw=1.2, ls=ARM_STYLE[arm], marker=".", ms=3)
            ax.set_ylabel(label if c else f"{DATASET_LABEL[ds]}\n{label}")
            if r == len(DATASETS) - 1:
                ax.set_xlabel("số mẫu đã xử lý")
    axes[0][0].legend(fontsize=8)
    fig.suptitle(f"{algo} — tài nguyên dọc luồng")
    fig.tight_layout()
    plt.show()

## 6. Đánh đổi chất lượng – tốc độ

Mỗi điểm là một (hướng, dataset), trung bình qua seed. Góc trên bên phải là tốt nhất: ARI cao mà vẫn nhanh.

In [ ]:
MARKERS = dict(zip(DATASETS, ["o", "s", "^"]))
agg = ok.groupby(["algo", "arm", "dataset"])[["ari", "throughput_sps"]].mean().reset_index()
fig, axes = plt.subplots(1, len(ALGOS), figsize=(6.5 * len(ALGOS), 4.5), squeeze=False)
for ax, algo in zip(axes[0], ALGOS):
    for row in agg[agg.algo == algo].itertuples():
        ax.scatter(row.throughput_sps, row.ari, color=ARM_COLOR[row.arm],
                   marker=MARKERS[row.dataset], s=60, edgecolor="k", lw=0.4)
    ax.set_xscale("log")
    ax.set_xlabel("thông lượng (mẫu/giây, log)")
    ax.set_ylabel("ARI")
    ax.set_title(algo)
handles = [plt.Line2D([], [], color=ARM_COLOR[a], marker="o", ls="", label=ARM_LABEL[a]) for a in ARMS]
handles += [plt.Line2D([], [], color="k", marker=MARKERS[d], ls="", mfc="none", label=DATASET_LABEL[d])
            for d in DATASETS]
axes[0][-1].legend(handles=handles, fontsize=8, loc="best")
fig.tight_layout()
plt.show()

## 7. Bảng tổng hợp

In [ ]:
cols = ["ari", "nmi", "purity", "throughput_sps", "runtime_s", "peak_memory_mb",
        "max_micro_clusters", "n_predicted_clusters"]
table = (ok.groupby(["algo", "arm_label", "dataset"], observed=True)[cols].mean()
         .rename(index=DATASET_LABEL, level="dataset"))
display(table.round({"ari": 4, "nmi": 4, "purity": 4, "throughput_sps": 0, "runtime_s": 1,
                     "peak_memory_mb": 3, "max_micro_clusters": 0, "n_predicted_clusters": 1}))

n = ok.n_samples.dropna().unique()
display(Markdown(
    f"Độ dài luồng thực tế: {', '.join(f'{int(v):,}' for v in sorted(n))} mẫu · "
    f"seed {SEEDS} · bộ lấy từ `{Path(FINAL_SETS_FILE).name}` · kết quả ở `{RESULTS}`"))

## 8. Đọc kết quả thế nào

- **Trong mẫu.** Cả ba dataset đều đã dùng để chọn bộ, nên bảng này ưu ái mọi hướng tune. Để so hướng với nhau một cách trung thực, dùng ước lượng LODO trong `report_50000.md`, và mục **3b** của báo cáo đó (thắng/hoà/thua theo nhiễu seed, headroom, trần có ràng buộc chi phí) thay cho trung bình ARI.
- **ARI 0,000 của DenStream không phải "gộp về một cụm".** Ở `epsilon` nhỏ DenStream không sinh ra p-micro-cluster nào và `predict_one` trả sentinel `-1` suốt luồng (RIVER_BUGS.md BUG-6); `-1` đó được chấm như một nhãn cụm. Dấu hiệu: `n_predicted_clusters` ≈ 2, `largest_cluster_share` = 1,000, `homogeneity` = 0. Hệ quả khi đọc bảng:
  - `0,000` **không** tốt hơn một ARI âm của cấu hình thực sự có gom cụm;
  - **thông lượng của những run đó là tốc độ của việc không làm gì** — default DenStream đạt hàng chục nghìn mẫu/s trên covtype/insects/keystroke vì nó giữ đúng 1 micro-cluster. Không đọc nó như một ưu điểm về tốc độ.
  - Mục 4i của `report_50000.md` liệt kê đầy đủ các ô suy biến.
- **So chi phí phải ở cùng số micro-cluster.** `predict_one` của DenStream chạy lại toàn bộ DBSCAN mỗi mẫu (GAP-3), chi phí ~ P^1,2…P^1,5; CluStream chỉ dựng lại pha offline mỗi `time_gap`. Ở cùng số micro-cluster trên covtype, DenStream chậm hơn CluStream 3,3× (P≈15) đến 15,6× (P≈60). Thông lượng thô so giữa hai thuật toán vì thế so hai kiến trúc, không so hai bộ tham số.
- **CluStream chỉ tune được chi phí, không tune được ARI.** Kiểm 4a của báo cáo xếp lưới ladder của nó là FAIL (biên độ / nhiễu = 2,5 < 3,0); riêng trên kdd99 tỉ số là **1,1**, tức toàn bộ chênh lệch giữa các hướng nằm trong nhiễu seed. Phần thắng thật của nó là `q` nhỏ hơn: nhanh hơn và tốn ít bộ nhớ hơn, và điều đó tất định.
- **Độ dài khác lúc tune.** Bộ được chọn ở 50.000 mẫu đầu; chạy toàn bộ dataset có thể cho thứ tự khác. Chênh lệch đó chính là điều notebook này kiểm.
- **Nhãn không đồng nghĩa với cấu trúc cụm.** covtype có đủ nhãn 7 lớp nhưng trần của cả lưới chỉ 0,019 — 7 loại rừng không phải 7 cụm trong không gian Euclid thô. **KDDCup99** bị một chiều chi phối (`src_bytes` > 99% phương sai). Nếu bảng có **shuttle**: nhãn của nó trong River là **nhị phân** (bình thường/bất thường), không phải 7 lớp gốc. Nếu có **electricity**: 2 lớp, tự tương quan mạnh. Cả hai chỉ dùng làm đối chứng phụ.
- **P2R** dùng mức độ mịn mà quy tắc đã đọc trên đường cong lúc tune (50.000 mẫu, seed 1); nó không đọc lại đường cong ở độ dài mới. Và tiêu chí không nhãn không phải lúc nào cũng dùng được: mục 4h của báo cáo đo regret của nó trên từng dataset, và trên shuttle `matr_trace_win` bỏ lỡ 0,278 trên một trần 0,297.
- **Số seed.** `SEEDS` hiện là 5. Đọc vạch min–max trước khi đọc chênh lệch giữa hai cột; và bảng winner's curse (4c của báo cáo) cho biết bao nhiêu phần của một chênh lệch là may mắn hoán vị.
